# Preprosessering av stemmedata

Kjør **Run All**. All kode ligger i notebooken og krever bare Pythons standardbibliotek. Rådata beholdes uendret.


In [ ]:
"""Prepare real_votes.csv without exposing evaluation labels to the model."""

import csv
import json
from collections import Counter
from datetime import date
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/dirty/real_votes.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Start notebooken fra prosjektmappen eller en undermappe.")


## Kolonner

Input og fasit holdes separat. CSV-verdier leses som tekst for å bevare nuller i ticker og ID-er.


In [ ]:
INPUT_COLUMNS = [
    "meeting_id", "item_no", "company_id", "company", "ticker", "country",
    "market", "meeting_date", "proposal_number", "proposal_text",
    "proposal_type", "proponent", "mgmt_rec",
]
LABEL_COLUMNS = ["meeting_id", "item_no", "nbim_vote"]
VOTES = {"FOR", "MOT", "AVSTÅ"}



## Rensing, filtrering og lagring

Funksjonen validerer nøkler og datoer, fjerner mellomrom og identiske duplikater, og normaliserer stemmer. Rader uten NBIM-stemme eller ledelsesanbefaling fjernes fra alle utdata. Den lagrer inputs.csv, labels.csv, splits.csv og quality_report.json.


In [ ]:
def prepare(source=ROOT / "data/dirty/real_votes.csv",
            output=ROOT / "data/prepared", holdout_year=2026):
    """Keep only proposals with both a vote and management recommendation."""
    source, output = Path(source), Path(output)
    with source.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        required = set(INPUT_COLUMNS + LABEL_COLUMNS)
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"Missing columns: {sorted(missing)}")
        original = list(reader)

    rows, seen, keys = [], set(), set()
    duplicates = 0
    for line, raw in enumerate(original, start=2):
        row = {key: (value or "").strip() for key, value in raw.items()}
        signature = tuple(sorted(row.items()))
        if signature in seen:
            duplicates += 1
            continue
        seen.add(signature)
        key = row["meeting_id"], row["item_no"]
        if not all(key) or key in keys:
            raise ValueError(f"Missing or conflicting meeting/item key at line {line}: {key}")
        keys.add(key)
        date.fromisoformat(row["meeting_date"])
        if not row["proposal_text"]:
            raise ValueError(f"Missing proposal text at line {line}")
        for column in ("mgmt_rec", "nbim_vote"):
            row[column] = row[column].upper()
            if row[column] and row[column] not in VOTES:
                raise ValueError(f"Unknown {column} at line {line}: {row[column]}")
        rows.append(row)

    missing_votes = sum(not row["nbim_vote"] for row in rows)
    missing_recommendations = sum(not row["mgmt_rec"] for row in rows)
    complete_rows = [row for row in rows if row["nbim_vote"] and row["mgmt_rec"]]
    missing_removed = len(rows) - len(complete_rows)
    rows = complete_rows
    rows.sort(key=lambda row: (row["meeting_date"], row["meeting_id"], row["item_no"]))
    metadata = []
    for row in rows:
        year = date.fromisoformat(row["meeting_date"]).year
        metadata.append({
            "meeting_id": row["meeting_id"], "item_no": row["item_no"],
            "split": "holdout" if year >= holdout_year else "development",
            "evaluation_eligible": str(bool(row["nbim_vote"])).lower(),
        })

    report = {
        "source": str(source), "source_rows": len(original),
        "output_rows": len(rows), "duplicates_removed": duplicates,
        "rows_removed_missing_vote_or_recommendation": missing_removed,
        "source_missing_nbim_vote": missing_votes,
        "source_missing_mgmt_rec": missing_recommendations,
        "missing_nbim_vote": sum(not row["nbim_vote"] for row in rows),
        "missing_mgmt_rec": sum(not row["mgmt_rec"] for row in rows),
        "vote_counts": dict(Counter(row["nbim_vote"] or "MISSING" for row in rows)),
        "holdout_start_year": holdout_year,
        "split_counts": dict(Counter(row["split"] for row in metadata)),
    }
    output.mkdir(parents=True, exist_ok=True)
    for filename, columns, records in [
        ("inputs.csv", INPUT_COLUMNS, rows),
        ("labels.csv", LABEL_COLUMNS, rows),
        ("splits.csv", ["meeting_id", "item_no", "split", "evaluation_eligible"], metadata),
    ]:
        with (output / filename).open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(records)
    (output / "quality_report.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    return report



## Kjør preprosesseringen

2026 og senere er holdout; tidligere år er development. Arkitekturplanens skille mellom eldre historikk og 2025 er ikke lagt inn i denne splitten. Eksisterende genererte filer overskrives.


In [ ]:
report = prepare(holdout_year=2026)
print(json.dumps(report, ensure_ascii=False, indent=2))


## Kontroller resultatet og lag selskapsoversikt


In [ ]:
output = ROOT / "data/prepared"
def read_csv(name):
    with (output / name).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

inputs = read_csv("inputs.csv")
labels = read_csv("labels.csv")
splits = read_csv("splits.csv")
keys = lambda rows: [(r["meeting_id"], r["item_no"]) for r in rows]
assert keys(inputs) == keys(labels) == keys(splits)
assert all(r["mgmt_rec"] for r in inputs)
assert all(r["nbim_vote"] for r in labels)
assert not {"nbim_vote", "against_board", "rationale_text", "rationale_section", "position_paper", "nbim_vote_raw"}.intersection(inputs[0] if inputs else INPUT_COLUMNS)

columns = ["company_id", "company", "ticker", "country", "market"]
companies = {r["company_id"]: {k: r[k] for k in columns} for r in inputs}
with (output / "company_tickers.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns)
    writer.writeheader()
    writer.writerows(sorted(companies.values(), key=lambda r: r["company"].casefold()))
print(f"Kontrollert {len(inputs)} forslag og lagret {len(companies)} selskaper.")
inputs[:5]
